In [1]:
# GOLD NOTEBOOK — Full Auto Discovery + Dim/Fact Builder + Audit
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, LongType, BooleanType
from delta.tables import DeltaTable
from datetime import datetime
import uuid

spark.conf.set("spark.sql.shuffle.partitions", "2")
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

# ── Base Config — ONLY EDIT THESE 3 LINES ────────────────────────────────────
SILVER_DB = "LH_Silver_layer.dbo"
GOLD_DB   = "LH_Gold_layer.dbo"
AUDIT_DB  = "LH_Gold_layer.audit"
# ─────────────────────────────────────────────────────────────────────────────

run_id        = str(uuid.uuid4())
pipeline_name = "Gold_Pipeline"
master_start  = datetime.now()

print(f"[GOLD] Started   : {master_start}")
print(f"[GOLD] Run ID    : {run_id}")
print(f"[GOLD] Silver DB : {SILVER_DB}")
print(f"[GOLD] Gold DB   : {GOLD_DB}")

# ── Silver tracking columns — these must be dropped before writing to Gold ────
# These were added by the Silver notebook and are internal to Silver layer only
SILVER_COLS_TO_DROP = [
    "_silver_sk",
    "_silver_effective_from",
    "_silver_effective_to",
    "_silver_is_current",
    "_silver_run_id",
]

# ── Fact table keyword detection ──────────────────────────────────────────────
# If a Silver table name contains ANY of these words → it is a FACT table
# Everything else → DIMENSION table
FACT_KEYWORDS = [
    "order", "sales", "transaction", "detail",
    "invoice", "payment", "purchase", "shipment",
    "receipt", "lineitem", "item"
]

# ── Audit Schemas (identical to Bronze and Silver) ────────────────────────────
AUDIT_SCHEMA = StructType([
    StructField("ROW_ID",        StringType(),    True),
    StructField("RUN_ID",        StringType(),    True),
    StructField("CREATED_DATE",  TimestampType(), True),
    StructField("PIPELINE_NAME", StringType(),    True),
    StructField("SOURCE_TYPE",   StringType(),    True),
    StructField("SOURCE_TABLE",  StringType(),    True),
    StructField("TARGET_TABLE",  StringType(),    True),
    StructField("START_TIME",    TimestampType(), True),
    StructField("END_TIME",      TimestampType(), True),
    StructField("STATUS",        StringType(),    True),
    StructField("STATUS_DESC",   StringType(),    True),
])

COUNT_SCHEMA = StructType([
    StructField("ROW_ID",                             StringType(),    True),
    StructField("RUN_ID",                             StringType(),    True),
    StructField("SOURCE_TYPE",                        StringType(),    True),
    StructField("SOURCE_TABLE",                       StringType(),    True),
    StructField("TARGET_TABLE",                       StringType(),    True),
    StructField("LAYER",                              StringType(),    True),
    StructField("SOUREC_FILE_COUNT",                  StringType(),    True),
    StructField("STAGING_FILE_COUNT",                 StringType(),    True),
    StructField("SOURCE_TABLE_COUNT",                 LongType(),      True),
    StructField("STAGING_TABLE_COUNT",                LongType(),      True),
    StructField("ERROR_COUNT_SOURCE_TO_STAGING_FILE", LongType(),      True),
    StructField("ERROR_COUNT_SOURCE_TO_STAGING",      LongType(),      True),
    StructField("Current_time",                       TimestampType(), True),
])

# ── Helper: Table Exists ──────────────────────────────────────────────────────
def tbl_exists(t):
    try:
        spark.sql(f"DESCRIBE TABLE {t}")
        return True
    except:
        return False

# ── Helper: Audit Log ─────────────────────────────────────────────────────────
def audit(tbl, table_name, src, tgt, t0, s, d=""):
    row = spark.createDataFrame(
        [(str(uuid.uuid4()), run_id, datetime.now(), pipeline_name,
          src, table_name, tgt, t0, datetime.now(), str(s), str(d))],
        schema=AUDIT_SCHEMA
    )
    for field in AUDIT_SCHEMA.fields:
        row = row.withColumn(field.name, F.col(field.name).cast(field.dataType))
    row.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(tbl)

# ── Helper: Count Log ─────────────────────────────────────────────────────────
def count_audit(table_name, src, tgt, src_cnt, stg_cnt, err=0):
    row = spark.createDataFrame(
        [(str(uuid.uuid4()), run_id, src, table_name, tgt, "GOLD",
          None, None, int(src_cnt), int(stg_cnt), int(err), int(err), datetime.now())],
        schema=COUNT_SCHEMA
    )
    for field in COUNT_SCHEMA.fields:
        row = row.withColumn(field.name, F.col(field.name).cast(field.dataType))
    row.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(f"{AUDIT_DB}.count_log_table")

# ── Helper: Detect Table Type ─────────────────────────────────────────────────
def detect_table_type(table_name):
    """
    Returns 'FACT' if the table name contains any FACT_KEYWORDS.
    Returns 'DIM' otherwise.
    Detection is case-insensitive.
    """
    name_lower = table_name.lower()
    for keyword in FACT_KEYWORDS:
        if keyword in name_lower:
            return "FACT"
    return "DIM"

# ── Helper: Drop Silver Internal Columns ─────────────────────────────────────
def drop_silver_cols(df):
    """
    Removes all Silver tracking columns that must not appear in Gold.
    Only drops columns that actually exist in the DataFrame.
    """
    cols_to_drop = [c for c in SILVER_COLS_TO_DROP if c in df.columns]
    return df.drop(*cols_to_drop)

# ── Helper: Auto-detect Primary Key ──────────────────────────────────────────
def detect_pk(columns):
    """
    Same logic as Bronze and Silver: first column ending with 'id'.
    Falls back to first column if none found.
    """
    return next((c for c in columns if c.lower().endswith("id")), columns[0])

# ── Build Dimension Table ─────────────────────────────────────────────────────
def build_dim(table_name, silver_df, pk_col, src_cnt):
    """
    Creates or replaces a Gold Dimension table.

    What it does:
      1. Drops all Silver internal columns
      2. Adds a surrogate key (_gold_sk) — unique integer per row
      3. Adds _gold_load_time timestamp
      4. Overwrites the Gold Dim table completely (Gold = current snapshot)

    Output table name: Dim_<TableName>  e.g. Dim_Customer
    """
    TGT      = f"{GOLD_DB}.Dim_{table_name}"
    SRC      = f"{SILVER_DB}.{table_name}"

    print(f"  [{table_name}] Type          : DIMENSION → {TGT}")

    # Drop Silver tracking columns
    df = drop_silver_cols(silver_df)

    # Add Gold surrogate key (monotonically_increasing_id = unique bigint per row)
    df = df.withColumn("_gold_sk",        F.monotonically_increasing_id())
    df = df.withColumn("_gold_load_time", F.current_timestamp())
    df = df.withColumn("_gold_run_id",    F.lit(run_id))

    stg_cnt = df.count()
    print(f"  [{table_name}] Rows to write : {stg_cnt}")

    # Always OVERWRITE — Gold Dim always reflects the current active Silver state
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TGT)

    # Optimize on PK for fast lookups
    spark.sql(f"OPTIMIZE {TGT} ZORDER BY ({pk_col})")

    tgt_cnt = spark.sql(f"SELECT COUNT(1) AS c FROM {TGT}").collect()[0]["c"]
    print(f"  [{table_name}] Target rows   : {tgt_cnt}")
    return TGT, stg_cnt, tgt_cnt

# ── Build Fact Table ──────────────────────────────────────────────────────────
def build_fact(table_name, silver_df, pk_col, src_cnt):
    """
    Creates or replaces a Gold Fact table.

    What it does:
      1. Drops all Silver internal columns
      2. Adds _gold_load_time timestamp
      3. Overwrites the Gold Fact table completely

    Output table name: Fact_<TableName>  e.g. Fact_SalesOrderHeader
    """
    TGT = f"{GOLD_DB}.Fact_{table_name}"
    SRC = f"{SILVER_DB}.{table_name}"

    print(f"  [{table_name}] Type          : FACT → {TGT}")

    # Drop Silver tracking columns
    df = drop_silver_cols(silver_df)

    # Add Gold audit columns
    df = df.withColumn("_gold_load_time", F.current_timestamp())
    df = df.withColumn("_gold_run_id",    F.lit(run_id))

    stg_cnt = df.count()
    print(f"  [{table_name}] Rows to write : {stg_cnt}")

    # Always OVERWRITE
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TGT)

    # Optimize on PK
    spark.sql(f"OPTIMIZE {TGT} ZORDER BY ({pk_col})")

    tgt_cnt = spark.sql(f"SELECT COUNT(1) AS c FROM {TGT}").collect()[0]["c"]
    print(f"  [{table_name}] Target rows   : {tgt_cnt}")
    return TGT, stg_cnt, tgt_cnt

# ── Core: Process One Table ───────────────────────────────────────────────────
def process_table(table_name):
    SRC = f"{SILVER_DB}.{table_name}"
    t0  = datetime.now()
    status  = "SUCCESS"
    desc    = ""
    src_cnt = 0
    stg_cnt = 0
    tgt_cnt = 0
    TGT     = ""

    print(f"\n  [{table_name}] Reading from : {SRC}")

    # ── Step 1: Read Silver — ONLY current/active records ─────────────────────
    # _silver_is_current = True  means this is the latest version of the record
    # Historical (expired) records are excluded from Gold
    try:
        silver_df = spark.read.format("delta").table(SRC)

        # Check if Silver has _silver_is_current column (SCD2 tables have it)
        if "_silver_is_current" in silver_df.columns:
            silver_df = silver_df.filter(F.col("_silver_is_current") == True)
            print(f"  [{table_name}] Filter        : _silver_is_current = True (SCD2 table)")
        else:
            # SCD1 tables — all rows are current, no filter needed
            print(f"  [{table_name}] Filter        : None (SCD1 table — all rows current)")

        src_cnt = silver_df.count()
        print(f"  [{table_name}] Source rows   : {src_cnt}")
        print(f"  [{table_name}] Columns       : {silver_df.columns}")

    except Exception as e:
        audit(f"{AUDIT_DB}.copy_audit_log",     table_name, SRC, "UNKNOWN", t0, "FAILED", str(e))
        audit(f"{AUDIT_DB}.notebook_audit_log", table_name, SRC, "UNKNOWN", t0, "FAILED", str(e))
        raise

    # ── Step 2: Zero Row Check ────────────────────────────────────────────────
    if src_cnt == 0:
        audit(f"{AUDIT_DB}.copy_audit_log",     table_name, SRC, "UNKNOWN", t0, "NO_DATA", "No active rows in Silver")
        count_audit(table_name, SRC, "UNKNOWN", 0, 0)
        audit(f"{AUDIT_DB}.notebook_audit_log", table_name, SRC, "UNKNOWN", t0, "NO_DATA", "No active rows in Silver")
        print(f"  [{table_name}] NO_DATA — skipping")
        return "NO_DATA"

    # ── Step 3: Auto-detect PK ────────────────────────────────────────────────
    pk_col = detect_pk(silver_df.columns)
    print(f"  [{table_name}] PK detected   : {pk_col}")

    # ── Step 4: Auto-classify as Dim or Fact and build Gold table ─────────────
    table_type = detect_table_type(table_name)
    print(f"  [{table_name}] Table type    : {table_type}")

    try:
        if table_type == "DIM":
            TGT, stg_cnt, tgt_cnt = build_dim(table_name, silver_df, pk_col, src_cnt)
        else:
            TGT, stg_cnt, tgt_cnt = build_fact(table_name, silver_df, pk_col, src_cnt)

    except Exception as e:
        status = "FAILED"
        desc   = str(e)
        audit(f"{AUDIT_DB}.copy_audit_log",     table_name, SRC, TGT, t0, status, desc)
        count_audit(table_name, SRC, TGT, src_cnt, stg_cnt, err=1)
        audit(f"{AUDIT_DB}.notebook_audit_log", table_name, SRC, TGT, t0, status, desc)
        raise

    # ── Step 5: Write Audit Logs ──────────────────────────────────────────────
    audit(f"{AUDIT_DB}.copy_audit_log",     table_name, SRC, TGT, t0, status, desc)
    count_audit(table_name, SRC, TGT, src_cnt, stg_cnt)
    audit(f"{AUDIT_DB}.notebook_audit_log", table_name, SRC, TGT, t0, status, desc)

    print(f"  [{table_name}] ✔ SUCCESS | src={src_cnt} stg={stg_cnt} tgt={tgt_cnt} | type={table_type}")
    return "SUCCESS"

# ── Auto-Discover ALL Tables from Silver Layer ────────────────────────────────
# Reads whatever Silver produced — Dimensions and Facts discovered automatically
# No table names hardcoded anywhere
silver_tables = [
    row.tableName
    for row in spark.sql(f"SHOW TABLES IN {SILVER_DB}").collect()
]

# ── Sort: Process Dimensions FIRST, then Facts ────────────────────────────────
# This ensures Dim tables exist before any Fact tables reference them
dims  = [t for t in silver_tables if detect_table_type(t) == "DIM"]
facts = [t for t in silver_tables if detect_table_type(t) == "FACT"]
ordered_tables = dims + facts

print(f"\n[GOLD] Tables discovered   : {len(silver_tables)}")
print(f"[GOLD] Dimensions          : {len(dims)}")
print(f"[GOLD] Facts               : {len(facts)}")
print(f"\n[GOLD] Processing order (Dims first, then Facts):")
for t in ordered_tables:
    ttype = detect_table_type(t)
    prefix = "Dim_" if ttype == "DIM" else "Fact_"
    print(f"           [{ttype:4}]  {t:40}  →  {prefix}{t}")

# ── Run All Tables ────────────────────────────────────────────────────────────
results = []
for idx, table in enumerate(ordered_tables, 1):
    ttype = detect_table_type(table)
    print(f"\n[GOLD] ══════════════════════════════════════════════")
    print(f"[GOLD] [{idx} / {len(ordered_tables)}]  {table}  [{ttype}]")
    print(f"[GOLD] ══════════════════════════════════════════════")
    try:
        s = process_table(table)
        results.append((table, ttype, s, ""))
    except Exception as e:
        results.append((table, ttype, "FAILED", str(e)))
        print(f"  [{table}] ✘ FAILED → {e}")

# ── Final Summary ─────────────────────────────────────────────────────────────
success = sum(1 for r in results if r[2] == "SUCCESS")
failed  = sum(1 for r in results if r[2] == "FAILED")
no_data = sum(1 for r in results if r[2] == "NO_DATA")

print(f"\n[GOLD] ══════════════════════════════════════════════════")
print(f"[GOLD] PIPELINE COMPLETE : {datetime.now()}")
print(f"[GOLD] Total   = {len(results)}")
print(f"[GOLD] Success = {success}")
print(f"[GOLD] Failed  = {failed}")
print(f"[GOLD] No Data = {no_data}")
print(f"[GOLD] ══════════════════════════════════════════════════")
for table, ttype, status, err in results:
    icon   = "✔" if status == "SUCCESS" else ("−" if status == "NO_DATA" else "✘")
    prefix = "Dim_" if ttype == "DIM" else "Fact_"
    msg    = f"→ {err[:80]}" if err else ""
    print(f"  {icon}  [{ttype:4}]  {table:40}  →  {prefix}{table:40}  {status}  {msg}")

if failed:
    raise Exception(f"[GOLD] {failed} table(s) failed. See details above.")

StatementMeta(, 3abc39b5-fad8-432d-8c33-42a609181d4f, 3, Finished, Cancelled, Cancelled, False)

[GOLD] Started   : 2026-04-13 02:37:01.217165
[GOLD] Run ID    : bc2a39dd-5d79-4098-970d-dd1b775c1815
[GOLD] Silver DB : LH_Silver_layer.dbo
[GOLD] Gold DB   : LH_Gold_layer.dbo

[GOLD] Tables discovered   : 17
[GOLD] Dimensions          : 15
[GOLD] Facts               : 2

[GOLD] Processing order (Dims first, then Facts):
           [DIM ]  product                                   →  Dim_product
           [DIM ]  productcategory                           →  Dim_productcategory
           [DIM ]  productcosthistory                        →  Dim_productcosthistory
           [DIM ]  productdescription                        →  Dim_productdescription
           [DIM ]  productdocument                           →  Dim_productdocument
           [DIM ]  productinventory                          →  Dim_productinventory
           [DIM ]  productlistpricehistory                   →  Dim_productlistpricehistory
           [DIM ]  productmodel                              →  Dim_productmodel

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/tmp/ipykernel_6376/2363012966.py", line 230, in process_table
    silver_df = spark.read.format("delta").table(SRC)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/spark/python/lib/pyspark.zip/pyspark/sql/readwriter.py", line 484, in table
    return self._df(self._jreader.table(tableName))
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages/py4j/java_gateway.py", line 1322, in __call__
    return_value = get_return_value(
                   ^^^^^^^^^^^^^^^^^
  File "/opt/spark/python/lib/pyspark.zip/pyspark/errors/exceptions/captured.py", line 179, in deco
    return f(*a, **kw)
           ^^^^^^^^^^^
  File "/home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages/py4j/protocol.py", line 326, in get_return_value
    raise Py4JJavaError(
py4j.protocol.Py4JJavaErro

In [2]:
# GOLD NOTEBOOK — Full Auto Discovery + Dim/Fact Builder + Audit
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, LongType, BooleanType
from delta.tables import DeltaTable
from datetime import datetime
import uuid

spark.conf.set("spark.sql.shuffle.partitions", "2")
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

# ── ONLY EDIT THESE 3 LINES ───────────────────────────────────────────────────
SILVER_DB = "LH_Silver_layer.dbo"    # your Silver lakehouse name
GOLD_DB   = "LH_Gold_layer.dbo"      # your Gold lakehouse name (dbo schema)
AUDIT_DB  = "LH_Gold_layer.audit"    # matches audit schema in screenshot
# ─────────────────────────────────────────────────────────────────────────────

run_id        = str(uuid.uuid4())
pipeline_name = "Gold_Pipeline"
master_start  = datetime.now()

print(f"[GOLD] Started   : {master_start}")
print(f"[GOLD] Run ID    : {run_id}")
print(f"[GOLD] Silver DB : {SILVER_DB}")
print(f"[GOLD] Gold DB   : {GOLD_DB}")
print(f"[GOLD] Audit DB  : {AUDIT_DB}")

# ── Silver tracking columns to drop before writing to Gold ───────────────────
SILVER_COLS_TO_DROP = [
    "_silver_sk", "_silver_effective_from", "_silver_effective_to",
    "_silver_is_current", "_silver_run_id",
]

# ── Fact keyword detection ────────────────────────────────────────────────────
FACT_KEYWORDS = [
    "order", "sales", "transaction", "detail",
    "invoice", "payment", "purchase", "shipment",
    "receipt", "lineitem", "item"
]

# ── Audit Schemas ─────────────────────────────────────────────────────────────
AUDIT_SCHEMA = StructType([
    StructField("ROW_ID",        StringType(),    True),
    StructField("RUN_ID",        StringType(),    True),
    StructField("CREATED_DATE",  TimestampType(), True),
    StructField("PIPELINE_NAME", StringType(),    True),
    StructField("SOURCE_TYPE",   StringType(),    True),
    StructField("SOURCE_TABLE",  StringType(),    True),
    StructField("TARGET_TABLE",  StringType(),    True),
    StructField("START_TIME",    TimestampType(), True),
    StructField("END_TIME",      TimestampType(), True),
    StructField("STATUS",        StringType(),    True),
    StructField("STATUS_DESC",   StringType(),    True),
])

COUNT_SCHEMA = StructType([
    StructField("ROW_ID",                             StringType(),    True),
    StructField("RUN_ID",                             StringType(),    True),
    StructField("SOURCE_TYPE",                        StringType(),    True),
    StructField("SOURCE_TABLE",                       StringType(),    True),
    StructField("TARGET_TABLE",                       StringType(),    True),
    StructField("LAYER",                              StringType(),    True),
    StructField("SOUREC_FILE_COUNT",                  StringType(),    True),
    StructField("STAGING_FILE_COUNT",                 StringType(),    True),
    StructField("SOURCE_TABLE_COUNT",                 LongType(),      True),
    StructField("STAGING_TABLE_COUNT",                LongType(),      True),
    StructField("ERROR_COUNT_SOURCE_TO_STAGING_FILE", LongType(),      True),
    StructField("ERROR_COUNT_SOURCE_TO_STAGING",      LongType(),      True),
    StructField("Current_time",                       TimestampType(), True),
])

# ── Helpers ───────────────────────────────────────────────────────────────────
def tbl_exists(t):
    try:
        spark.sql(f"DESCRIBE TABLE {t}")
        return True
    except:
        return False

def audit(tbl, table_name, src, tgt, t0, s, d=""):
    try:
        row = spark.createDataFrame(
            [(str(uuid.uuid4()), run_id, datetime.now(), pipeline_name,
              src, table_name, tgt, t0, datetime.now(), str(s), str(d)[:500])],
            schema=AUDIT_SCHEMA
        )
        for field in AUDIT_SCHEMA.fields:
            row = row.withColumn(field.name, F.col(field.name).cast(field.dataType))
        row.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(tbl)
    except Exception as ae:
        print(f"  [AUDIT WARNING] {ae}")

def count_audit(table_name, src, tgt, src_cnt, stg_cnt, err=0):
    try:
        row = spark.createDataFrame(
            [(str(uuid.uuid4()), run_id, src, table_name, tgt, "GOLD",
              None, None, int(src_cnt), int(stg_cnt), int(err), int(err), datetime.now())],
            schema=COUNT_SCHEMA
        )
        for field in COUNT_SCHEMA.fields:
            row = row.withColumn(field.name, F.col(field.name).cast(field.dataType))
        row.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(f"{AUDIT_DB}.count_log_table")
    except Exception as ce:
        print(f"  [COUNT WARNING] {ce}")

def detect_table_type(table_name):
    name_lower = table_name.lower()
    for keyword in FACT_KEYWORDS:
        if keyword in name_lower:
            return "FACT"
    return "DIM"

def drop_silver_cols(df):
    cols_to_drop = [c for c in SILVER_COLS_TO_DROP if c in df.columns]
    return df.drop(*cols_to_drop)

def detect_pk(columns):
    return next((c for c in columns if c.lower().endswith("id")), columns[0])

# ── Build Dimension Table ─────────────────────────────────────────────────────
def build_dim(table_name, silver_df, pk_col, src_cnt):
    TGT = f"{GOLD_DB}.Dim_{table_name}"
    SRC = f"{SILVER_DB}.{table_name}"
    print(f"  [{table_name}] Type          : DIMENSION → {TGT}")

    df = drop_silver_cols(silver_df)
    df = df.withColumn("_gold_sk",        F.monotonically_increasing_id())
    df = df.withColumn("_gold_load_time", F.current_timestamp())
    df = df.withColumn("_gold_run_id",    F.lit(run_id))

    stg_cnt = df.count()
    print(f"  [{table_name}] Rows to write : {stg_cnt}")

    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TGT)
    spark.sql(f"OPTIMIZE {TGT} ZORDER BY (`{pk_col}`)")
    tgt_cnt = spark.sql(f"SELECT COUNT(1) AS c FROM {TGT}").collect()[0]["c"]
    print(f"  [{table_name}] Target rows   : {tgt_cnt}")
    return TGT, stg_cnt, tgt_cnt

# ── Build Fact Table ──────────────────────────────────────────────────────────
def build_fact(table_name, silver_df, pk_col, src_cnt):
    TGT = f"{GOLD_DB}.Fact_{table_name}"
    SRC = f"{SILVER_DB}.{table_name}"
    print(f"  [{table_name}] Type          : FACT → {TGT}")

    df = drop_silver_cols(silver_df)
    df = df.withColumn("_gold_load_time", F.current_timestamp())
    df = df.withColumn("_gold_run_id",    F.lit(run_id))

    stg_cnt = df.count()
    print(f"  [{table_name}] Rows to write : {stg_cnt}")

    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TGT)
    spark.sql(f"OPTIMIZE {TGT} ZORDER BY (`{pk_col}`)")
    tgt_cnt = spark.sql(f"SELECT COUNT(1) AS c FROM {TGT}").collect()[0]["c"]
    print(f"  [{table_name}] Target rows   : {tgt_cnt}")
    return TGT, stg_cnt, tgt_cnt

# ── Core: Process One Table ───────────────────────────────────────────────────
def process_table(table_name):
    SRC = f"{SILVER_DB}.{table_name}"
    t0  = datetime.now()
    status  = "SUCCESS"
    desc    = ""
    src_cnt = 0
    stg_cnt = 0
    tgt_cnt = 0
    TGT     = ""

    print(f"\n  [{table_name}] Reading from : {SRC}")

    # ── Step 1: Read Silver — current records only ────────────────────────────
    try:
        silver_df = spark.read.format("delta").table(SRC)

        if "_silver_is_current" in silver_df.columns:
            silver_df = silver_df.filter(F.col("_silver_is_current") == True)
            print(f"  [{table_name}] Filter        : _silver_is_current = True (SCD2)")
        else:
            print(f"  [{table_name}] Filter        : None (SCD1 — all rows are current)")

        src_cnt = silver_df.count()
        print(f"  [{table_name}] Source rows   : {src_cnt}")
        print(f"  [{table_name}] Columns       : {silver_df.columns}")

    except Exception as e:
        audit(f"{AUDIT_DB}.copy_audit_log",     table_name, SRC, "UNKNOWN", t0, "FAILED", str(e))
        audit(f"{AUDIT_DB}.notebook_audit_log", table_name, SRC, "UNKNOWN", t0, "FAILED", str(e))
        raise

    # ── Step 2: Zero Row Check ────────────────────────────────────────────────
    if src_cnt == 0:
        audit(f"{AUDIT_DB}.copy_audit_log",     table_name, SRC, "UNKNOWN", t0, "NO_DATA", "No active rows in Silver")
        count_audit(table_name, SRC, "UNKNOWN", 0, 0)
        audit(f"{AUDIT_DB}.notebook_audit_log", table_name, SRC, "UNKNOWN", t0, "NO_DATA", "No active rows in Silver")
        print(f"  [{table_name}] NO_DATA — skipping")
        return "NO_DATA"

    # ── Step 3: Auto-detect PK ────────────────────────────────────────────────
    pk_col     = detect_pk(silver_df.columns)
    table_type = detect_table_type(table_name)
    print(f"  [{table_name}] PK detected   : {pk_col}")
    print(f"  [{table_name}] Table type    : {table_type}")

    # ── Step 4: Build Gold table ──────────────────────────────────────────────
    try:
        if table_type == "DIM":
            TGT, stg_cnt, tgt_cnt = build_dim(table_name, silver_df, pk_col, src_cnt)
        else:
            TGT, stg_cnt, tgt_cnt = build_fact(table_name, silver_df, pk_col, src_cnt)

    except Exception as e:
        status = "FAILED"
        desc   = str(e)
        audit(f"{AUDIT_DB}.copy_audit_log",     table_name, SRC, TGT, t0, status, desc)
        count_audit(table_name, SRC, TGT, src_cnt, stg_cnt, err=1)
        audit(f"{AUDIT_DB}.notebook_audit_log", table_name, SRC, TGT, t0, status, desc)
        raise

    # ── Step 5: Audit Logs ────────────────────────────────────────────────────
    audit(f"{AUDIT_DB}.copy_audit_log",     table_name, SRC, TGT, t0, status, desc)
    count_audit(table_name, SRC, TGT, src_cnt, stg_cnt)
    audit(f"{AUDIT_DB}.notebook_audit_log", table_name, SRC, TGT, t0, status, desc)

    print(f"  [{table_name}] ✔ SUCCESS | src={src_cnt} stg={stg_cnt} tgt={tgt_cnt} | type={table_type}")
    return "SUCCESS"

# ── Auto-Discover ALL Tables from Silver Layer ────────────────────────────────
silver_tables = [
    row.tableName
    for row in spark.sql(f"SHOW TABLES IN {SILVER_DB}").collect()
]

dims  = [t for t in silver_tables if detect_table_type(t) == "DIM"]
facts = [t for t in silver_tables if detect_table_type(t) == "FACT"]
ordered_tables = dims + facts   # Dimensions FIRST, then Facts

print(f"\n[GOLD] Tables in {SILVER_DB}: {len(silver_tables)}")
print(f"[GOLD] Dimensions : {len(dims)}")
print(f"[GOLD] Facts      : {len(facts)}")
print(f"\n[GOLD] Processing order:")
for t in ordered_tables:
    ttype  = detect_table_type(t)
    prefix = "Dim_" if ttype == "DIM" else "Fact_"
    print(f"           [{ttype:4}]  {t:40}  →  {prefix}{t}")

# ── Run All Tables ────────────────────────────────────────────────────────────
results = []
for idx, table in enumerate(ordered_tables, 1):
    ttype = detect_table_type(table)
    print(f"\n[GOLD] ══════════════════════════════════════════════")
    print(f"[GOLD] [{idx} / {len(ordered_tables)}]  {table}  [{ttype}]")
    print(f"[GOLD] ══════════════════════════════════════════════")
    try:
        s = process_table(table)
        results.append((table, ttype, s, ""))
    except Exception as e:
        results.append((table, ttype, "FAILED", str(e)[:200]))
        print(f"  [{table}] ✘ FAILED → {e}")

# ── Final Summary ─────────────────────────────────────────────────────────────
success = sum(1 for r in results if r[2] == "SUCCESS")
failed  = sum(1 for r in results if r[2] == "FAILED")
no_data = sum(1 for r in results if r[2] == "NO_DATA")

print(f"\n[GOLD] ══════════════════════════════════════════════════")
print(f"[GOLD] PIPELINE COMPLETE : {datetime.now()}")
print(f"[GOLD] Total   = {len(results)}")
print(f"[GOLD] Success = {success}")
print(f"[GOLD] Failed  = {failed}")
print(f"[GOLD] No Data = {no_data}")
print(f"[GOLD] ══════════════════════════════════════════════════")
for table, ttype, status, err in results:
    icon   = "✔" if status == "SUCCESS" else ("−" if status == "NO_DATA" else "✘")
    prefix = "Dim_" if ttype == "DIM" else "Fact_"
    msg    = f"→ {err[:80]}" if err else ""
    print(f"  {icon}  [{ttype:4}]  {table:40}  →  {prefix}{table:40}  {status}  {msg}")

if failed:
    raise Exception(f"[GOLD] {failed} table(s) failed. See details above.")


StatementMeta(, 3abc39b5-fad8-432d-8c33-42a609181d4f, 4, Finished, Available, Finished, False)

[GOLD] Started   : 2026-04-13 02:37:54.838221
[GOLD] Run ID    : 76edfe82-d9f3-47b1-80ba-377c89636916
[GOLD] Silver DB : LH_Silver_layer.dbo
[GOLD] Gold DB   : LH_Gold_layer.dbo
[GOLD] Audit DB  : LH_Gold_layer.audit

[GOLD] Tables in LH_Silver_layer.dbo: 17
[GOLD] Dimensions : 15
[GOLD] Facts      : 2

[GOLD] Processing order:
           [DIM ]  product                                   →  Dim_product
           [DIM ]  productcategory                           →  Dim_productcategory
           [DIM ]  productcosthistory                        →  Dim_productcosthistory
           [DIM ]  productdescription                        →  Dim_productdescription
           [DIM ]  productdocument                           →  Dim_productdocument
           [DIM ]  productinventory                          →  Dim_productinventory
           [DIM ]  productlistpricehistory                   →  Dim_productlistpricehistory
           [DIM ]  productmodel                              →  Dim_product

In [5]:
%%sql
select * from LH_Gold_layer.audit.copy_audit_log

StatementMeta(, 3abc39b5-fad8-432d-8c33-42a609181d4f, 7, Finished, Available, Finished, False)

<Spark SQL result set with 185 rows and 11 fields>